# Known-variance cEBMF

This notebook walks through the new `S=` argument on `cEBMF`, which lets you supply standard errors directly instead of having the model learn the noise variance. Two common cases:

1. **Z-scores** – pass `S=1.0` (or any scalar). Every observation is treated as having unit standard error.
2. **Per-entry SEs** – pass an `(N, P)` tensor (or anything broadcastable to that shape) when each observation has its own measurement standard error – e.g. an effect-size matrix from a burden test where rows are genes and columns are tissues.

When `S` is provided the variance is taken as truth and is *not* re-estimated during fitting. Bad entries (NaN, 0, negative) in `S` are folded into the missing-data mask, just like NaNs in `Y`.


In [1]:
import numpy as np
import torch

from cebmf_torch import cEBMF
from cebmf_torch.cebmf import NoiseType

torch.manual_seed(0)
rng = np.random.default_rng(0)

## A toy low-rank target with known noise
We generate $Y = LF^T + \epsilon$ with $\epsilon_{ij} \sim N(0, \sigma^2)$ and a known $\sigma$.

In [2]:
n, p, rank = 200, 100, 3
sigma = 0.4
L_true = rng.normal(size=(n, rank))
F_true = rng.normal(size=(p, rank))
truth = L_true @ F_true.T
Y = torch.tensor(truth + rng.normal(scale=sigma, size=(n, p)), dtype=torch.float32)
truth_t = torch.tensor(truth, dtype=torch.float32)

def rmse(a, b):
    return float(torch.sqrt(((a - b) ** 2).mean()))

print(f'data shape: {tuple(Y.shape)}, sigma = {sigma}')

data shape: (200, 100), sigma = 0.4


### Case 1 — Z-score case: `S = 1.0`
If your matrix is already z-scored you typically just want $S = 1$. Pass it in and the noise variance is fixed (the noise type is forced to `KNOWN`).

In [3]:
Z = (Y - Y.mean()) / Y.std()  # crude z-scoring just to make the demo concrete
model_z = cEBMF(Z, K=5, S=1.0)
print('noise type:', model_z.noise.type)
print('tau_map[0,0] =', float(model_z.tau_map[0, 0]))  # = 1 / S^2 = 1
fit_z = model_z.fit(maxit=20)
print(f'final K after pruning: {fit_z.L.shape[1]}')

noise type: known
tau_map[0,0] = 1.0


c:\Users\willi\miniconda3\envs\cebmf\Lib\site-packages\torch\utils\_contextlib.py:124: UserWarning: Factors not initialized; using SVD initialization.
  return func(*args, **kwargs)


final K after pruning: 3


### Case 2 — Known per-entry SE (a matrix)
If you have a measurement standard error per observation (typical for effect-size estimates), pass that matrix as `S`. The model never tries to estimate the noise — it uses your `S` exactly as supplied.

In [4]:
S_mat = torch.full_like(Y, sigma)            # constant in this toy example
model_mat = cEBMF(Y, K=5, S=S_mat)
fit_mat = model_mat.fit(maxit=20)
fitted = fit_mat.L @ fit_mat.F.T
print(f'truth-fit RMSE: {rmse(fitted, truth_t):.4f}')
print(f'noise level   : {sigma:.4f}')

truth-fit RMSE: 0.0834
noise level   : 0.4000


### Case 3 — Per-column known SE (e.g. per-tissue measurement quality)
`S` may be any tensor broadcastable to `(N, P)`. A `(P,)` vector applies the same SE to every row but a different one per column — e.g. each tissue has its own standard error in a burden-test matrix.

In [5]:
se_per_col = torch.tensor(np.linspace(0.2, 0.6, p), dtype=torch.float32)
Y_col = torch.tensor(truth + rng.normal(scale=se_per_col.numpy(), size=(n, p)), dtype=torch.float32)
model_col = cEBMF(Y_col, K=5, S=se_per_col)
print('S broadcast to:', tuple(model_col.S.shape))
fit_col = model_col.fit(maxit=20)
print(f'truth-fit RMSE: {rmse(fit_col.L @ fit_col.F.T, truth_t):.4f}')

S broadcast to: (200, 100)
truth-fit RMSE: 0.0770


### Case 4 — NaN handling in S
Entries of `S` that are NaN, 0, or negative are folded into the missing-data mask. A warning is raised only when those entries align with values that were observed in `Y`.

In [6]:
import warnings
S_bad = S_mat.clone()
S_bad[0, 0] = float('nan')   # bad SE at an observed entry
S_bad[1, 1] = 0.0            # 0 SE -> infinite precision, treat as missing
with warnings.catch_warnings():
    warnings.simplefilter('always')
    model_bad = cEBMF(Y, K=3, S=S_bad)
print('mask at (0,0):', float(model_bad.mask[0, 0]))  # 0 -> dropped
print('mask at (1,1):', float(model_bad.mask[1, 1]))  # 0 -> dropped
print('mask elsewhere stays at 1:', float(model_bad.mask[5, 5]))

mask at (0,0): 0.0
mask at (1,1): 0.0
mask elsewhere stays at 1: 1.0


C:\Document\Serieux\Travail\python_work\package\cebmf_torch\src\cebmf_torch\cebmf\cebmf.py:513: UserWarning: S has 2 non-finite or non-positive entries at observed positions in `data`; these entries will be treated as missing.
  self._setup_known_variance()


### Case 5 — Verifying the noise really is held fixed
When `S` is supplied, `update_tau()` is a no-op. `tau_map` should not change as we iterate.

In [7]:
model_check = cEBMF(Y, K=3, S=S_mat)
model_check.initialise_factors('svd')
tau_before = model_check.tau_map.clone()
for _ in range(5):
    model_check.iter_once()
print('tau_map unchanged across iterations:',
      torch.equal(model_check.tau_map, tau_before))

tau_map unchanged across iterations: True
